# PN2: fixed-budget prime-survival bridge

## tl;dr

ARA did **not** beat the established probabilistic baselines on the untouched `100,000,000-110,000,000` target. The primary Information^3 candidate model lost to the p29-conditioned prime-number-theorem baseline by `0.000160973` bits/candidate. The primary ARA edge model lost to the p29-conditioned Hardy-Littlewood baseline by `0.000036725` bits/edge. Both 95% block-bootstrap intervals are wholly negative. An independent reconstruction passed `476/476` checks.

The result establishes a useful boundary: ARA maps the deterministic primorial-wheel geometry, but the tested local ARA coordinates do not yet add prime-survival information beyond established analytic baselines.


## Context & Methods

The predictor receives only candidates that survive sieving through prime 29. It must estimate which of those candidates are genuinely prime in a later, untouched interval. The p31 PN1H wheel target is not generated or inspected.

For a candidate at location (n), the analytic candidate baseline is

\[
p_{\mathrm{PNT29}}(n)=\frac{1}{\log n\prod_{q\leq29}(1-1/q)}.
\]

For an adjacent p29-wheel edge of gap (g), the pair baseline uses the corresponding conditional Hardy-Littlewood probability. Competing fitted models use raw local gaps, a four-gap stencil, plain ARA, an Information^3-style ARA stencil, and a decompressed ARA representation.

All models were fitted only on `[10,000,000,20,000,000)`. The primary bin count (`12`), shrinkage (`64`), endpoints, baselines and 40-block bootstrap were frozen before the target was opened. Lower log loss is better; reported deltas are `baseline loss - ARA loss`, so positive values favour ARA.


## Data

- Development interval: `[10,000,000,20,000,000)`
- Untouched target: `[100,000,000,110,000,000)`
- Sieve budget: primes through `29`
- Target p29-wheel candidates: `1,579,479`
- Surviving primes: `541,854` (`34.3059%`)
- Adjacent candidate edges: `1,579,478`
- Edges with two prime endpoints: `184,913` (`11.7072%`)


## Results

### 1. Rerun the frozen target analysis


In [1]:
import json
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
import pn2_prime_survival_bridge as analysis
analysis.run_target()
results = json.loads((HERE / "PN2_RESULTS.json").read_text(encoding="utf-8"))
print("Status:", results["status"])
print("Prime-31 wheel accessed:", results["pn1h_p31_wheel_accessed"])
print("Candidate endpoint:", results["primary_candidate_endpoint"])
print("Edge endpoint:", results["primary_edge_endpoint"])


{
  "status": "TARGET_COMPLETE",
  "candidate_endpoint": {
    "observed_delta_bits": -0.0001609725889273263,
    "bootstrap_lower_95_bits": -0.00019108274980260696,
    "bootstrap_upper_95_bits": -0.00012991811127640682,
    "positive_block_share": 0.05,
    "baseline_model": "candidate_pnt29",
    "ara_model": "ara_i3_b12_l64",
    "support": false
  },
  "edge_endpoint": {
    "observed_delta_bits": -3.67246432446444e-05,
    "bootstrap_lower_95_bits": -5.0887348336825675e-05,
    "bootstrap_upper_95_bits": -2.2377259495247524e-05,
    "positive_block_share": 0.15,
    "baseline_model": "edge_hl29",
    "ara_model": "ara_edge_b12_l64",
    "support": false
  },
  "candidate_events": 1579479,
  "edge_events": 1579478,
  "p31_wheel_accessed": false
}
Status: TARGET_COMPLETE
Prime-31 wheel accessed: False
Candidate endpoint: {'observed_delta_bits': -0.0001609725889273263, 'bootstrap_lower_95_bits': -0.00019108274980260696, 'bootstrap_upper_95_bits': -0.00012991811127640682, 'positive_b

![Primary model and block comparisons](PN2_SURVIVAL_MODEL_COMPARISON.png)

Both primary ARA deltas are negative in most target blocks, and both bootstrap intervals exclude zero in the direction favouring the analytic baseline.


### 2. Inspect the primary scores and the small plain-ARA sensitivity


In [2]:
scores = pd.read_csv(HERE / "PN2_MODEL_SCORES.csv")
selected = [
    "candidate_pnt29", "raw_local_l64", "raw_stencil_l64",
    "ara_plain_b12_l64", "ara_i3_b12_l64", "ara_decompressed_b12_l64",
    "edge_hl29", "raw_edge_l64", "ara_edge_b12_l64",
    "ara_edge_decompressed_b12_l64",
]
view = scores[scores.model.isin(selected)][
    ["task", "model", "log_loss_bits", "gain_vs_analytic_bits", "calibration_error"]
].copy()
print(view.to_string(index=False, float_format=lambda value: f"{value:.12f}"))

plain = scores[(scores.task == "candidate") & scores.model.str.startswith("ara_plain_b") & scores.model.str.endswith("_l64")]
print("\nPlain-ARA bin sensitivity")
print(plain[["model", "log_loss_bits", "gain_vs_analytic_bits"]].to_string(index=False, float_format=lambda value: f"{value:.12f}"))
assert (plain.gain_vs_analytic_bits > 0).sum() == 1


     task                         model  log_loss_bits  gain_vs_analytic_bits  calibration_error
candidate               candidate_pnt29 0.927715513769         0.000000000000    -0.000256644643
     edge                     edge_hl29 0.520887305414         0.000000000000    -0.000437956843
candidate                 raw_local_l64 0.927742231125        -0.000026717356    -0.000225250076
candidate               raw_stencil_l64 0.928070887537        -0.000355373768    -0.000187587205
candidate             ara_plain_b12_l64 0.927714742034         0.000000771735    -0.000232414220
candidate                ara_i3_b12_l64 0.927876486357        -0.000160972589    -0.000224361049
candidate      ara_decompressed_b12_l64 0.927985149490        -0.000269635721    -0.000194576881
     edge                  raw_edge_l64 0.521020824701        -0.000133519288    -0.000588595745
     edge              ara_edge_b12_l64 0.520924030057        -0.000036724643    -0.000615085543
     edge ara_edge_decompresse

The 12-bin plain ARA candidate model gains only `0.000000772` bits/candidate over PNT29, about `1.2` bits across the entire target. The same representation loses with 8, 16 and 24 bins, and the frozen Information^3 primary loses clearly. This isolated sensitivity is therefore not treated as support.


### 3. Check gap-class frequency and location calibration


In [3]:
print("Gap-class summaries")
for model, metrics in results["gap_class_frequency"].items():
    if isinstance(metrics, dict):
        print(model, metrics)
print("\nLocation summaries")
for model, metrics in results["location_calibration"].items():
    print(model, metrics)


Gap-class summaries
hl29 {'poisson_deviance': 20.71434714836209, 'weighted_absolute_relative_error': 0.007356362539107945}
raw_edge {'poisson_deviance': 36.21408405919499, 'weighted_absolute_relative_error': 0.00959169841400268}
ara_edge {'poisson_deviance': 36.118842375797286, 'weighted_absolute_relative_error': 0.010115650738462696}
ara_edge_decompressed {'poisson_deviance': 36.3849412700731, 'weighted_absolute_relative_error': 0.009666130976078418}

Location summaries
pnt29 {'mean_absolute_percentage_error': 0.0020876377679138725, 'signed_calibration_error_share': -0.0007481070999185722}
raw_stencil {'mean_absolute_percentage_error': 0.0021579812069315958, 'signed_calibration_error_share': -0.0005468079045635097}
ara_i3 {'mean_absolute_percentage_error': 0.0021127137124149066, 'signed_calibration_error_share': -0.0006540019374715926}
ara_decompressed {'mean_absolute_percentage_error': 0.0021319525072257656, 'signed_calibration_error_share': -0.0005671824839858522}


![Gap-class residuals](PN2_GAP_CLASS_RESIDUALS.png)

Hardy-Littlewood has the lowest gap-class Poisson deviance (`20.714`) and weighted absolute relative error (`0.7356%`). PNT29 also has the best 20-block location MAPE (`0.2088%`). The local fitted models track the broad pattern but do not improve it.


### 4. Run the independent full-target reconstruction


In [4]:
import pn2_independent_validator as validator
validator.main()
validation = json.loads((HERE / "PN2_INDEPENDENT_VALIDATION.json").read_text(encoding="utf-8"))
print("Independent validation:", validation["status"])
print("Checks:", validation["passed_check_count"], "/", validation["check_count"])
assert validation["status"] == "PASS"
assert validation["passed_check_count"] == validation["check_count"] == 476
assert validation["pn1h_p31_wheel_accessed"] is False


{
  "status": "PASS",
  "passed_check_count": 476,
  "check_count": 476
}
Independent validation: PASS
Checks: 476 / 476


## Takeaways

1. This first direct prime-survival bridge is a clean negative result for the frozen ARA endpoints.
2. The p29-conditioned PNT and Hardy-Littlewood baselines are already extremely well calibrated on the target.
3. The exact mapped-log-ratio ARA control reproduces its equivalent ordinary ratio with zero numerical difference; it is a coordinate crosswalk, not additional predictive information.
4. The deterministic PN1 wheel results remain valid, but they do not automatically transfer to predicting which wheel candidates survive all later prime factors.
5. Further work should not tune new bins on this target. A new endpoint requires a fresh frozen interval and a structural reason for the added information.

Full interpretation is in `PN2_PRIME_SURVIVAL_BRIDGE_REPORT.md`.
